# Step 8 — YOLOv8n vs YOLOv8s Benchmark

**Runs on Colab/Kaggle GPU ONLY** — this project's local dev machine is CPU-only; training never runs there. This notebook trains both YOLOv8n and YOLOv8s on the same 10-class PPE dataset with identical hyperparameters, then compares them on **mAP, CPU inference speed, and model size** so the "why nano for the real-time/live-stream use case" argument in `training/benchmark.md` is backed by real numbers, not intuition.

**Before running:** in Colab, set **Runtime -> Change runtime type -> GPU** (a T4 is fine).

## One-time setup you need to do first
This dataset (`data/css-data/`, ~162MB) is gitignored and was never committed — it has to be moved into your Google Drive once:
1. Locally, zip the `data/css-data` folder into `css-data.zip`.
2. Upload `css-data.zip` to your Google Drive at `MyDrive/ppe_detector/css-data.zip`.

(If you'd rather use a different Drive path, just edit `DRIVE_ZIP_PATH` in the cell below.)

## Judgment calls made explicit (no silent defaults)

- **Drive-mount + local unzip, not training directly off Drive**: Drive is network-mounted inside Colab — thousands of small per-image reads during training would be far slower than reading from Colab's local `/content` disk. The dataset is unzipped to `/content/data` once, then training reads from there.
- **`epochs=100`, `patience=20`**: 100 matches what a prior public run on this *exact* dataset used (bundled in the Kaggle mirror as `results_yolov8n_100e`) — not an arbitrary pick. `patience=20` (early stopping) protects against wasting GPU time if a model converges early or plateaus.
- **Same `imgsz=640`, `batch=16`, `seed=42` for BOTH models** — a benchmark is only meaningful if the only thing that differs between the two runs is the model architecture itself.
- **All 10 dataset classes trained as-is, not filtered down to the 2 we act on (NO-Hardhat/NO-Safety Vest)** — consistent with the earlier decision: extra classes cost the model a bit of output-head capacity but don't hurt what we care about, and filtering label files ourselves would be unnecessary engineering risk.
- **Inference-speed benchmark forces `device='cpu'`, even though this notebook runs on a GPU runtime.** Training on GPU is necessary (this machine has none), but our deployment target is CPU-only — reporting GPU inference speed would make the eventual nano-vs-small argument dishonest for our actual use case.

In [ ]:
!nvidia-smi

In [ ]:
# Pin to the same Ultralytics version used locally (requirements.txt) so
# training/export behavior matches what the rest of this project expects.
!pip install -q ultralytics==8.3.40

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DRIVE_ZIP_PATH = '/content/drive/MyDrive/ppe_detector/css-data.zip'
LOCAL_DATA_DIR = '/content/data'

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
!unzip -q -n "{DRIVE_ZIP_PATH}" -d "{LOCAL_DATA_DIR}"
!ls "{LOCAL_DATA_DIR}"

## Reconstructing `data.yaml`

This Kaggle mirror ships **no `data.yaml` of its own**. The class order below was NOT guessed — it was read directly from a prior training run's bundled config (`data/results_yolov8n_100e/kaggle/working/ppe_data.yaml`) and cross-checked against label-file class-ID frequency counts locally (class 5 was by far the most frequent, consistent with it being "Person"). This is the SAME order already confirmed and wired into `config/config.yaml` (`person_class: 5`, `NO-Hardhat: 2`, `NO-Safety Vest: 4`) — keeping this notebook's class order and the running app's class order in sync is what makes weights trained here actually usable by the rest of the system without a remap.

In [ ]:
import glob
import yaml

DATA_YAML_PATH = '/content/ppe_data.yaml'


def find_data_root(base_dir):
    """Auto-detect where train/valid/test actually landed after unzipping.
    NOT hardcoded to '{base_dir}/css-data' — zip tools differ on whether
    they preserve a wrapping folder (css-data/train/images/...) or extract
    the split folders directly (train/images/...) depending on how the
    source folder was selected/compressed locally. Searching for wherever
    'train/images' actually ended up avoids re-guessing the layout."""
    matches = glob.glob(f'{base_dir}/**/train/images', recursive=True)
    if not matches:
        raise RuntimeError(
            f"Couldn't find a 'train/images' folder anywhere under {base_dir}. "
            f"Run '!find {base_dir} -maxdepth 4' in a cell to inspect the actual layout."
        )
    train_images_dir = matches[0]
    return os.path.dirname(os.path.dirname(train_images_dir))  # parent of 'train'


DATA_ROOT = find_data_root(LOCAL_DATA_DIR)
print('Detected DATA_ROOT:', DATA_ROOT)

# Order confirmed against config/config.yaml — do not reorder without
# re-confirming both files together.
CLASS_NAMES = [
    'Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest',
    'Person', 'Safety Cone', 'Safety Vest', 'machinery', 'vehicle',
]

data_yaml = {
    'train': f'{DATA_ROOT}/train/images',
    'val': f'{DATA_ROOT}/valid/images',
    'test': f'{DATA_ROOT}/test/images',
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES,
}

with open(DATA_YAML_PATH, 'w') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(open(DATA_YAML_PATH).read())

In [ ]:
# Sanity check before spending GPU time training on it.
for split in ('train', 'valid', 'test'):
    img_dir = f'{DATA_ROOT}/{split}/images'
    n = len(os.listdir(img_dir)) if os.path.isdir(img_dir) else 0
    print(f'{split}: {n} images')

## Train YOLOv8n

`device=0` targets the first (only) GPU Colab assigns. `seed=42` for reproducibility. Results (weights, curves, metrics) land under `/content/runs/benchmark/yolov8n/`.

In [ ]:
from ultralytics import YOLO

model_n = YOLO('yolov8n.pt')
results_n = model_n.train(
    data=DATA_YAML_PATH,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    seed=42,
    device=0,
    project='/content/runs/benchmark',
    name='yolov8n',
)

## Train YOLOv8s

Identical hyperparameters to the nano run above — the ONLY thing that should differ is the base model.

In [ ]:
model_s = YOLO('yolov8s.pt')
results_s = model_s.train(
    data=DATA_YAML_PATH,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    seed=42,
    device=0,
    project='/content/runs/benchmark',
    name='yolov8s',
)

## Validate both on the held-out TEST split

Deliberately `split='test'`, not the `valid` split used for early-stopping decisions during training — reporting accuracy on data the training loop already used to make decisions (even indirectly, via `patience`) would overstate real generalization.

In [ ]:
n_best = '/content/runs/benchmark/yolov8n/weights/best.pt'
s_best = '/content/runs/benchmark/yolov8s/weights/best.pt'

metrics_n = YOLO(n_best).val(data=DATA_YAML_PATH, split='test')
metrics_s = YOLO(s_best).val(data=DATA_YAML_PATH, split='test')

print('YOLOv8n  mAP50:', metrics_n.box.map50, ' mAP50-95:', metrics_n.box.map)
print('YOLOv8s  mAP50:', metrics_s.box.map50, ' mAP50-95:', metrics_s.box.map)

## CPU inference-speed benchmark

Forces `device='cpu'` regardless of this notebook running on a GPU runtime — see the judgment-calls cell above for why. A few warm-up predictions are discarded before timing (first inference pays a one-time initialization cost that a running service wouldn't repeat).

In [ ]:
import time
import glob

def benchmark_cpu_ms_per_image(weights_path, n_images=30, warmup=5):
    model = YOLO(weights_path)
    sample_images = glob.glob(f'{DATA_ROOT}/test/images/*.jpg')[: n_images + warmup]

    for img in sample_images[:warmup]:
        model.predict(source=img, device='cpu', imgsz=640, verbose=False)

    timed_images = sample_images[warmup:]
    start = time.perf_counter()
    for img in timed_images:
        model.predict(source=img, device='cpu', imgsz=640, verbose=False)
    elapsed = time.perf_counter() - start

    return (elapsed / len(timed_images)) * 1000.0  # ms/image

cpu_ms_n = benchmark_cpu_ms_per_image(n_best)
cpu_ms_s = benchmark_cpu_ms_per_image(s_best)
print(f'YOLOv8n CPU: {cpu_ms_n:.1f} ms/image')
print(f'YOLOv8s CPU: {cpu_ms_s:.1f} ms/image')

## Model size and parameter count

In [ ]:
def size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)

def param_count_millions(weights_path):
    model = YOLO(weights_path)
    return sum(p.numel() for p in model.model.parameters()) / 1e6

size_n, size_s = size_mb(n_best), size_mb(s_best)
params_n, params_s = param_count_millions(n_best), param_count_millions(s_best)
print(f'YOLOv8n: {size_n:.1f} MB, {params_n:.1f}M params')
print(f'YOLOv8s: {size_s:.1f} MB, {params_s:.1f}M params')

In [ ]:
results_rows = [
    {
        'model': 'YOLOv8n',
        'params_M': round(params_n, 1),
        'size_MB': round(size_n, 1),
        'mAP50': round(float(metrics_n.box.map50), 4),
        'mAP50-95': round(float(metrics_n.box.map), 4),
        'cpu_ms_per_image': round(cpu_ms_n, 1),
    },
    {
        'model': 'YOLOv8s',
        'params_M': round(params_s, 1),
        'size_MB': round(size_s, 1),
        'mAP50': round(float(metrics_s.box.map50), 4),
        'mAP50-95': round(float(metrics_s.box.map), 4),
        'cpu_ms_per_image': round(cpu_ms_s, 1),
    },
]


def to_markdown_table(rows):
    # Hand-rolled instead of pandas' DataFrame.to_markdown(), which needs
    # the `tabulate` package — not guaranteed preinstalled on every Colab
    # base image, and not worth risking a failure here after 2+ hours of
    # GPU training just to format a 2-row table.
    headers = list(rows[0].keys())
    lines = [
        '| ' + ' | '.join(headers) + ' |',
        '|' + '|'.join(['---'] * len(headers)) + '|',
    ]
    for row in rows:
        lines.append('| ' + ' | '.join(str(row[h]) for h in headers) + ' |')
    return '\n'.join(lines)


md_table = to_markdown_table(results_rows)
print(md_table)

## Save results: weights to Drive, table to a markdown file

Colab's local disk (`/content`) is wiped when the runtime disconnects — anything worth keeping gets copied to Drive before the session ends.

In [ ]:
import shutil

OUT_DIR = '/content/drive/MyDrive/ppe_detector/weights_out'
os.makedirs(OUT_DIR, exist_ok=True)

shutil.copy(n_best, f'{OUT_DIR}/yolov8n_best.pt')
shutil.copy(s_best, f'{OUT_DIR}/yolov8s_best.pt')

with open(f'{OUT_DIR}/benchmark_results_table.md', 'w') as f:
    f.write(md_table)

print(md_table)
print(f'\nSaved to {OUT_DIR} — download both .pt files and benchmark_results_table.md from Drive.')

## Next steps (back on the local machine)

1. Download `yolov8n_best.pt` and `yolov8s_best.pt` from `MyDrive/ppe_detector/weights_out/` into this project's `weights/` folder.
2. Copy the printed markdown table into `training/benchmark.md`, replacing the placeholder numbers, and fill in the written nano-vs-small argument using these REAL numbers.
3. Report the actual table back — we'll decide together (with real numbers in hand) which weights `config.yaml`'s `detection.model_path` should point at, and write the final reasoning for why nano is (or isn't) the right call for the live-stream/CPU use case.